# Phân Tích Cảm Xúc Đa Khía Cạnh (Generative ABSA) - ViT5
Sử dụng mô hình `VietAI/vit5-base` để chuyển đổi bài toán trích xuất khía cạnh & cảm xúc thành dạng Text-to-Text.
Dữ liệu đầu vào: `Phân tích đánh giá: [TITLE] + [BODY]`
Đầu ra mục tiêu: `Tổng thể: tích cực | Content: tiêu cực | Packaging: trung tính`

## Import Thư viện Và Cấu Hình

In [1]:
# Gỡ bản PyTorch mặc định của Kaggle (tránh xung đột)
!pip uninstall -y torch torchvision torchaudio

# Ép version đồng bộ cho cả transformers, accelerate và peft
!pip install -q transformers==4.39.3 accelerate==0.28.0 peft==0.10.0 sentencepiece

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 3.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.7 MB/s eta 0:00:00:00:0100

In [ ]:
import json, os, random, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq,
    set_seed
)
from torch.utils.data import Dataset

from huggingface_hub import hf_hub_download
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import EarlyStoppingCallback
import pandas as pd
import re
from sklearn.utils import resample
from transformers import EarlyStoppingCallback, Seq2SeqTrainingArguments, Seq2SeqTrainer

import torch.nn as nn
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType

from transformers import AutoTokenizer, DataCollatorForSeq2Seq
from torch.utils.data import Dataset

from sklearn.utils.class_weight import compute_class_weight

# Tắt cảnh báo TF/CUDA (giúp output sạch hơn, giải quyết các dòng E0000)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


# Thiết lập Random Seed
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Cấu hình tham số
MODEL_NAME = 'VietAI/vit5-base'
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 64
EPOCHS = 8
BATCH_SIZE = 8
LEARNING_RATE = 2e-4 

TRAIN_PATH = '/kaggle/input/datasets/jyang10/tiki-cleaned-book-reviews/train_clean.json'
TEST_PATH = '/kaggle/input/datasets/jyang10/tiki-cleaned-book-reviews/test_clean.json'
VAL_PATH=  '/kaggle/input/datasets/jyang10/tiki-cleaned-book-reviews/val_clean.json'

OUTPUT_DIR = './vit5_absa_results'

ASPECT_COLS = ['as_content', 'as_physical', 'as_price', 'as_packaging', 'as_delivery', 'as_service']
ASPECT_NAMES = ['Content', 'Physical', 'Price', 'Packaging', 'Delivery', 'Service']

# Từ điển ánh xạ nhãn
LABEL_2_TEXT = {0: 'tiêu cực', 1: 'trung tính', 2: 'tích cực'}
TEXT_2_LABEL = {'tiêu cực': 0, 'trung tính': 1, 'tích cực': 2}
ABSENT_CLASS = 3

print("Device:", torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

2026-03-30 06:13:40.541185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774851220.696344      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774851220.740649      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774851221.109322      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774851221.109358      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774851221.109361      55 computation_placer.cc:177] computation placer alr

Device: cuda


## Tiền Xử Lý và Chuẩn Bị Dữ Liệu Cho Mô Hình

In [ ]:
def format_target_text(row):
    """Chuyển đổi nhãn dạng số thành chuỗi Text-to-Text mục tiêu."""
    parts = []
    # Cảm xúc tổng thể
    if pd.notna(row['sentiment_llm']) and int(row['sentiment_llm']) in LABEL_2_TEXT:
        parts.append(f"Tổng thể: {LABEL_2_TEXT[int(row['sentiment_llm'])]}")
    else:
        parts.append(f"Tổng thể: {LABEL_2_TEXT[1]}") # default trung tính
        
    # Các khía cạnh
    for col, name in zip(ASPECT_COLS, ASPECT_NAMES):
        val = row[col]
        if pd.notna(val) and int(val) in LABEL_2_TEXT:
            parts.append(f"{name}: {LABEL_2_TEXT[int(val)]}")
            
    return ", ".join(parts)


def load_and_prepare_data(path, is_train=False):
    df = pd.read_json(path).copy()
    
    title = df.get('review_title', df.get('title', pd.Series(['']*len(df)))).fillna('').astype(str).str.strip()
    body = df.get('content', df.get('text', pd.Series(['']*len(df)))).fillna('').astype(str).str.strip()
    
    # CẢI TIẾN 1: Mở rộng định nghĩa Neutral trong Prompt
    instruction = (
        "Nhiệm vụ: Phân tích cảm xúc (tích cực, trung tính, tiêu cực) cho Tổng thể và các khía cạnh. "
        "Quy tắc quan trọng: Đánh giá 'trung tính' nếu câu mang tính chất trần thuật khách quan (không bộc lộ cảm xúc rõ ràng) "
        "HOẶC chứa cả ý khen lẫn chê ở mức độ tương đương nhau. "
        "Văn bản: "
    )
    
    df['input_text'] = instruction + title + ". " + body
    
    df['sentiment_llm'] = pd.to_numeric(df['sentiment_llm'], errors='coerce')
    df = df.dropna(subset=['sentiment_llm']).copy()
    
    for col in ASPECT_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(ABSENT_CLASS).astype(int).clip(0, ABSENT_CLASS)
        
    df['target_text'] = df.apply(format_target_text, axis=1)
    
    # CẢI TIẾN 2: Tính toán Sample Weights dựa trên Overall Sentiment (Chỉ áp dụng cho tập Train)
    if is_train:
        y_train = df['sentiment_llm'].values
        # Tính trọng số nghịch đảo: Class nào ít sẽ có trọng số cao
        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        weight_dict = {cls: weight for cls, weight in zip(np.unique(y_train), class_weights)}
        
        # Tạo cột weight cho từng dòng dữ liệu
        df['sample_weight'] = df['sentiment_llm'].map(weight_dict)
    else:
        df['sample_weight'] = 1.0 # Val và Test giữ nguyên trọng số là 1
    
    return df[['input_text', 'target_text', 'sentiment_llm', 'sample_weight'] + ASPECT_COLS].reset_index(drop=True)

# Khởi tạo dữ liệu
train_df = load_and_prepare_data(TRAIN_PATH, is_train=True)
val_df   = load_and_prepare_data(VAL_PATH, is_train=False)
test_df  = load_and_prepare_data(TEST_PATH, is_train=False)

## Tokenization

In [ ]:
print("Khởi tạo Tokenizer chuẩn...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class Seq2SeqABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=MAX_INPUT_LENGTH, max_target_len=MAX_TARGET_LENGTH):
        self.df = df
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        model_inputs = self.tokenizer(
            row['input_text'], 
            max_length=self.max_input_len, 
            truncation=True,
            padding=False
        )
        
        labels = self.tokenizer(
            text_target=row['target_text'], 
            max_length=self.max_target_len, 
            truncation=True,
            padding=False
        )
        
        model_inputs["labels"] = labels["input_ids"]
        # Thêm sample_weight vào dictionary (HuggingFace DataCollator sẽ tự động gom batch thành Tensor)
        model_inputs["sample_weights"] = row['sample_weight']
        
        return model_inputs

# Khởi tạo lại dataset
train_dataset = Seq2SeqABSADataset(train_df, tokenizer)
val_dataset = Seq2SeqABSADataset(val_df, tokenizer)
test_dataset = Seq2SeqABSADataset(test_df, tokenizer)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model if 'model' in locals() else None, padding="longest")

print("Khởi tạo Tokenizer và Dataset thành công!")

Khởi tạo Tokenizer chuẩn...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Khởi tạo Tokenizer và Dataset thành công!


## Metrics

In [5]:
def parse_generated_text(text):
    """Dịch ngược văn bản sinh ra thành mảng nhãn dạng số để tính Metrics."""
    res = {'sentiment_llm': 1} # Khởi tạo mặc định
    for c in ASPECT_COLS: res[c] = ABSENT_CLASS
        
    # CHÚ Ý SỬA Ở ĐÂY: Split bằng dấu phẩy
    parts = [p.strip() for p in text.split(',')]
    
    for p in parts:
        if ':' not in p: continue
        key, val = p.split(':', 1)
        key, val = key.strip(), val.strip().lower()
        
        label = TEXT_2_LABEL.get(val, -1)
        if label != -1:
            if key == 'Tổng thể': res['sentiment_llm'] = label
            elif key == 'Content': res['as_content'] = label
            elif key == 'Physical': res['as_physical'] = label
            elif key == 'Price': res['as_price'] = label
            elif key == 'Packaging': res['as_packaging'] = label
            elif key == 'Delivery': res['as_delivery'] = label
            elif key == 'Service': res['as_service'] = label
    return res
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    true_sent, pred_sent = [], []
    true_aspects, pred_aspects = [], []
    
    for d_pred, d_label in zip(decoded_preds, decoded_labels):
        p_dict = parse_generated_text(d_pred)
        l_dict = parse_generated_text(d_label)
        
        true_sent.append(l_dict['sentiment_llm'])
        pred_sent.append(p_dict['sentiment_llm'])
        
        true_aspects.append([l_dict[c] for c in ASPECT_COLS])
        pred_aspects.append([p_dict[c] for c in ASPECT_COLS])
        
    true_sent = np.array(true_sent)
    pred_sent = np.array(pred_sent)
    true_aspects = np.array(true_aspects)
    pred_aspects = np.array(pred_aspects)
    
    # Tính toán F1
    f1_sentiment = f1_score(true_sent, pred_sent, labels=[0, 1, 2], average='macro', zero_division=0)
    
    present_mask = true_aspects.flatten() != ABSENT_CLASS
    if present_mask.any():
        f1_aspect_present = f1_score(
            true_aspects.flatten()[present_mask],
            pred_aspects.flatten()[present_mask],
            labels=[0, 1, 2],
            average='macro',
            zero_division=0
        )
    else:
        f1_aspect_present = 0.0
        
    f1_final = 0.5 * f1_sentiment + 0.5 * f1_aspect_present
    
    return {
        "f1_sentiment": round(f1_sentiment, 4), 
        "f1_aspect_present": round(f1_aspect_present, 4),
        "f1_final": round(f1_final, 4)
    }

##  ViT5 & LoRA 

In [ ]:
print("Đang khởi tạo mô hình gốc ViT5...")
base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# CẢI TIẾN 3: Mở rộng LoRA target modules
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# ==========================================
# CẢI TIẾN 4: CUSTOM TRAINER XỬ LÝ WEIGHTED LOSS & EVALUATION KHÔNG BỊ LỖI
# ==========================================
class WeightedSeq2SeqTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Lấy nhãn và trọng số ra khỏi batch inputs
        labels = inputs.pop("labels")
        weights = inputs.pop("sample_weights", None) 
        
        # Forward pass lấy Logits
        outputs = model(**inputs)
        logits = outputs.logits 
        
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        loss = loss.view(logits.size(0), logits.size(1))
        mask = (labels != -100).float()
        
        loss_per_sequence = (loss * mask).sum(dim=1) / mask.sum(dim=1)
        
        if weights is not None:
            weights = weights.to(loss_per_sequence.device)
            loss_per_sequence = loss_per_sequence * weights
            
        loss = loss_per_sequence.mean()
        
        return (loss, outputs) if return_outputs else loss

    # FIX LỖI Ở ĐÂY: Can thiệp vào quá trình Prediction/Evaluation
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None, **gen_kwargs):
        # Lọc bỏ key 'sample_weights' (nếu có) trước khi đưa vào model.generate()
        filtered_inputs = {k: v for k, v in inputs.items() if k != "sample_weights"}
        
        # Trả lại dữ liệu sạch cho hàm gốc xử lý tiếp
        return super().prediction_step(
            model, 
            filtered_inputs, 
            prediction_loss_only, 
            ignore_keys=ignore_keys,
            **gen_kwargs
        )

# Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",  
    save_strategy="epoch",
    learning_rate=3e-4, 
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    weight_decay=0.01, 
    save_total_limit=2,
    num_train_epochs=EPOCHS,
    predict_with_generate=True, 
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=3,     
    load_best_model_at_end=True,     
    metric_for_best_model="f1_sentiment", 
    greater_is_better=True, 
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Khởi tạo Custom Trainer
trainer = WeightedSeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer, 
    data_collator=data_collator, 
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
)

print("Sẵn sàng huấn luyện với Weighted Custom Loss đã fix triệt để!")

Đang khởi tạo mô hình gốc ViT5...
trainable params: 5,013,504 || all params: 230,964,480 || trainable%: 2.1706818295176817
Sẵn sàng huấn luyện với Weighted Custom Loss đã fix triệt để!


## Training

In [9]:
print("Bắt đầu quá trình Fine-tuning ViT5 bằng LoRA...")
trainer.train()

# Lưu mô hình cuối cùng
final_model_dir = './vit5_absa_lora_final'
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)                                                       
print(f"Đã lưu mô hình (LoRA adapter) tại: {final_model_dir}")

Bắt đầu quá trình Fine-tuning ViT5 bằng LoRA...


Epoch,Training Loss,Validation Loss,F1 Sentiment,F1 Aspect Present,F1 Final
1,0.433900,0.189517,0.770800,0.558700,0.664700
2,0.191400,0.152246,0.805900,0.688000,0.746900
3,0.158800,0.141324,0.800500,0.648200,0.724400
4,0.131200,0.134234,0.823700,0.736300,0.780000
5,0.119800,0.129913,0.828200,0.768900,0.798600
6,0.091000,0.128631,0.824100,0.772100,0.798100
7,0.080600,0.128492,0.833400,0.779600,0.806500
8,0.073600,0.132026,0.829100,0.780900,0.805000


Đã lưu mô hình (LoRA adapter) tại: ./vit5_absa_lora_final


## Đánh giá

In [ ]:
def plot_confusion_matrices(true_sentiment: np.ndarray, pred_sentiment: np.ndarray, true_aspects: np.ndarray, pred_aspects: np.ndarray) -> None:
    # Tạo thư mục lưu ảnh nếu chưa có
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # ==========================================
    # 1. Vẽ Confusion Matrix cho Overall Sentiment
    # ==========================================
    plt.figure(figsize=(6, 5))
    sent_cm = confusion_matrix(true_sentiment, pred_sentiment, labels=[0, 1, 2])
    sns.heatmap(sent_cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
                xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'],
                annot_kws={"size": 14}) # Tăng kích thước chữ bên trong ma trận
    
    plt.title('Overall Sentiment', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/cm_overall_sentiment.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ==========================================
    # 2. Vẽ Confusion Matrix cho từng Khía cạnh (Aspects)
    # ==========================================
    for idx, (col, aspect_name) in enumerate(zip(ASPECT_COLS, ASPECT_NAMES)):
        # Bỏ qua các khía cạnh không xuất hiện (ABSENT_CLASS = 3) trong nhãn thực tế
        mask = true_aspects[:, idx] != ABSENT_CLASS
        if mask.sum() == 0:
            continue
            
        plt.figure(figsize=(6, 5))
        cm = confusion_matrix(true_aspects[:, idx][mask], pred_aspects[:, idx][mask], labels=[0, 1, 2])
        
        # Dùng colormap Oranges cho các khía cạnh để phân biệt với Overall (Blues)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False, 
                    xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'],
                    annot_kws={"size": 14})
        
        plt.title(f'Aspect: {aspect_name} (n={mask.sum()})', fontsize=16, fontweight='bold', pad=15)
        plt.xlabel('Predicted Label', fontsize=12)
        plt.ylabel('True Label', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/cm_aspect_{aspect_name.lower()}.png', dpi=150, bbox_inches='tight')
        plt.show()


# ==========================================
# THỰC THI ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
print("Đánh giá mô hình trên tập Test...")
raw_pred = trainer.predict(test_dataset)
print("Metrics từ Trainer:", raw_pred.metrics)

# 1. Giải mã (Decode) Token IDs sinh ra thành Văn bản tiếng Việt
preds = np.where(raw_pred.predictions != -100, raw_pred.predictions, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

true_sentiment, pred_sentiment = [], []
true_aspects, pred_aspects = [], []

# 2. Phân tích (Parse) văn bản thành nhãn số
for idx, row in test_df.iterrows():
    p_dict = parse_generated_text(decoded_preds[idx])
    
    true_sentiment.append(int(row['sentiment_llm']))
    pred_sentiment.append(p_dict['sentiment_llm'])
    
    true_aspects.append([int(row[c]) for c in ASPECT_COLS])
    pred_aspects.append([p_dict[c] for c in ASPECT_COLS])

true_sentiment = np.array(true_sentiment)
pred_sentiment = np.array(pred_sentiment)
true_aspects = np.array(true_aspects)
pred_aspects = np.array(pred_aspects)

# 3. In Báo cáo phân loại (Classification Report)
print('\n=========================================')
print('=== BÁO CÁO TỔNG THỂ (OVERALL SENTIMENT) ===')
print('=========================================')
print(classification_report(true_sentiment, pred_sentiment, labels=[0, 1, 2], target_names=['neg', 'neu', 'pos'], zero_division=0))

# ==========================================
# 5. BÁO CÁO TRUNG BÌNH TOÀN BỘ ASPECT (present only)
# ==========================================

all_true = []
all_pred = []

for idx in range(len(ASPECT_COLS)):
    t_asp = true_aspects[:, idx]
    p_asp = pred_aspects[:, idx]

    mask = t_asp != ABSENT_CLASS  # chỉ lấy các aspect có xuất hiện

    if mask.sum() == 0:
        continue

    all_true.extend(t_asp[mask])
    all_pred.extend(p_asp[mask])

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print('\n=========================================')
print('=== 6 ASPECTS')
print('=========================================')

print(classification_report(
    all_true,
    all_pred,
    labels=[0, 1, 2],
    target_names=['Negative', 'Neutral', 'Positive'],
    zero_division=0
))

print('\n=========================================')
print('=== BÁO CÁO CHI TIẾT TỪNG KHÍA CẠNH ===')
print('=========================================')
for idx, (col_name, aspect_name) in enumerate(zip(ASPECT_COLS, ASPECT_NAMES)):
    t_asp = true_aspects[:, idx]
    p_asp = pred_aspects[:, idx]
    
    mask = t_asp != ABSENT_CLASS
    
    print(f'\n--- Khía cạnh: {aspect_name.upper()} (n = {mask.sum()}) ---')
    
    if mask.sum() > 0:
        print(classification_report(
            t_asp[mask], 
            p_asp[mask], 
            labels=[0, 1, 2], 
            target_names=['neg', 'neu', 'pos'], 
            zero_division=0
        ))
    else:
        print(f"Không có dữ liệu thực tế cho khía cạnh này trong tập Test.")

Đánh giá mô hình trên tập Test...


Metrics từ Trainer: {'test_loss': 0.11956693977117538, 'test_f1_sentiment': 0.8405, 'test_f1_aspect_present': 0.779, 'test_f1_final': 0.8098, 'test_runtime': 185.7405, 'test_samples_per_second': 10.795, 'test_steps_per_second': 0.339}

=== BÁO CÁO TỔNG THỂ (OVERALL SENTIMENT) ===
              precision    recall  f1-score   support

         neg       0.95      0.92      0.93      1048
         neu       0.66      0.70      0.68       324
         pos       0.90      0.92      0.91       633

    accuracy                           0.88      2005
   macro avg       0.84      0.85      0.84      2005
weighted avg       0.89      0.88      0.88      2005


=== 6 ASPECTS
              precision    recall  f1-score   support

    Negative       0.90      0.76      0.83       805
     Neutral       0.71      0.52      0.60       410
    Positive       0.94      0.88      0.91      1291

   micro avg       0.89      0.79      0.84      2506
   macro avg       0.85      0.72      0.78      25